# Traffic FL — Colab Experiments

**Three runs:**
1. **Run 1** — QW + Hier(geo) + Adaptive (zero-fix, no time features)
2. **Run 2** — QW + Hier(geo) + Adaptive (zero-fix + time features)
3. **Run 3** — QW + Hier(DTW) + Adaptive (zero-fix + time features)

Run all cells from top to bottom.

## Setup

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Set project directory
import os
PROJECT_DIR = '/content/drive/MyDrive/traffic-fl'
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Check GPU
!nvidia-smi
import torch
print(f'\nPyTorch sees CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 5. Verify data is accessible
import numpy as np
proc = np.load('data/processed/metr_la_processed.npz')
print(f'Data loaded: X_train={proc["X_train"].shape}')
print(f'Has time features: {"tod_train" in proc}')
print(f'Clusters: {np.load("data/processed/dtw_clusters.npz")["cluster_labels"].max() + 1}')

---
## Run 1 — Zero-fix + QW + Hier(geo) + Adaptive (no time features)
This fills the missing ablation row AND shows the zero-masking fix impact.

In [ ]:
# Set USE_TIME_FEATURES = False for Run 1
config_path = 'config.py'
with open(config_path, 'r') as f:
    content = f.read()
content = content.replace('USE_TIME_FEATURES = True', 'USE_TIME_FEATURES = False')
with open(config_path, 'w') as f:
    f.write(content)
print('config.py updated: USE_TIME_FEATURES = False')

In [ ]:
# RUN 1: QW + Hier(geo) + Adaptive, speed-only (zero-fix applied)
!python scripts/run_fl_experiment.py \
  --mode clustered \
  --rounds 50 \
  --nodes all \
  --tf_start 0.0 \
  --local_steps 20 \
  --lr 0.0005 \
  --quality_agg \
  --hier_alpha 0.8 \
  --hier_every 5 \
  --hier_mode geo \
  --selection adaptive

---
## Run 2 — Zero-fix + Time Features + QW + Hier(geo) + Adaptive
Adds sin/cos time-of-day encoding (input_size=3). Geographic hier weights.

In [ ]:
# Set USE_TIME_FEATURES = True for Run 2
config_path = 'config.py'
with open(config_path, 'r') as f:
    content = f.read()
content = content.replace('USE_TIME_FEATURES = False', 'USE_TIME_FEATURES = True')
with open(config_path, 'w') as f:
    f.write(content)
print('config.py updated: USE_TIME_FEATURES = True')

In [ ]:
# RUN 2: QW + Hier(geo) + Adaptive, with time features
!python scripts/run_fl_experiment.py \
  --mode clustered \
  --rounds 50 \
  --nodes all \
  --tf_start 0.0 \
  --local_steps 20 \
  --lr 0.0005 \
  --quality_agg \
  --hier_alpha 0.8 \
  --hier_every 5 \
  --hier_mode geo \
  --selection adaptive

---
## Run 3 — DTW-Weighted Hierarchical Aggregation
Uses DTW pattern similarity instead of geographic distance for inter-cluster sharing.
Pattern-similar clusters share more, regardless of physical location.
Uses time features (already enabled from Run 2).

In [ ]:
# RUN 3: Same as Run 2 but with DTW-weighted hierarchical aggregation
!python scripts/run_fl_experiment.py \
  --mode clustered \
  --rounds 50 \
  --nodes all \
  --tf_start 0.0 \
  --local_steps 20 \
  --lr 0.0005 \
  --quality_agg \
  --hier_alpha 0.8 \
  --hier_every 5 \
  --hier_mode dtw \
  --selection adaptive

---
## Check Results

In [ ]:
# List all result files
import os
import numpy as np

print(f'{"File":<60} | {"MAE":>6} | {"Comm_MB":>9}')
print('-' * 85)
for f in sorted(os.listdir('results')):
    if f.endswith('.npz'):
        try:
            d = np.load(f'results/{f}', allow_pickle=True)
            mae = float(d['overall_mae'])
            comm = float(d['total_bytes_mb'])
            print(f'{f:<60} | {mae:>6.3f} | {comm:>9.1f}')
        except:
            print(f'{f:<60} | error')